In [7]:
import os
import glob
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.features import shapes
from shapely.geometry import shape, box

In [8]:
def process_coastlines(shape_dir, ref_raster, crs, buffs):
    # Define your raster values (Update these to match your raster's encoding)
    LAND_VALUE = 1  
    WATER_VALUE = 0 
    
    shp_files = glob.glob(os.path.join(shape_dir, "*.shp"))
    
    total_sar_length = 0.0
    buffer_stats = {dist: 0.0 for dist in buffs} # Will store sum of intersecting lengths
    
    with rasterio.open(ref_raster) as src:
        raster_crs = src.crs
        
        for shp_file in shp_files:
            print(f"\nProcessing {os.path.basename(shp_file)}...")
            
            # 1. Load SAR coastline shapefile
            gdf = gpd.read_file(shp_file)
            
            # Ensure it's in the target projected CRS for accurate meter measurements
            if gdf.crs != crs:
                gdf = gdf.to_crs(crs)
                
            # Compute Length of this specific shapefile
            shp_length = gdf.geometry.length.sum()
            total_sar_length += shp_length
            print(f"  SAR Coastline Length: {shp_length:.2f} m")
            
            # 2. Define a bounding box to clip the raster
            # We add a generous 50m buffer to the bounding box to ensure we capture 
            # reference land/water interfaces that might be just outside the SAR line
            bbox_geom = box(*gdf.total_bounds).buffer(50) 
            bbox_gdf_raster_crs = gpd.GeoDataFrame(geometry=[bbox_geom], crs=TARGET_CRS).to_crs(raster_crs)
            
            try:
                # 3. Clip the raster to the expanded bounding box of the shapefile
                out_image, out_transform = mask(src, bbox_gdf_raster_crs.geometry, crop=True)
                out_band = out_image[0]
                
                # 4. Extract the Land/Water Interface (edges) from the raster
                # Vectorize the clipped raster
                results = (
                    {'properties': {'val': v}, 'geometry': s}
                    for i, (s, v) in enumerate(shapes(out_band, transform=out_transform))
                )
                
                # Convert to GeoDataFrame
                geoms = list(results)
                if not geoms:
                    print("  No raster data found in this extent.")
                    continue
                    
                raster_gdf = gpd.GeoDataFrame.from_features(geoms, crs=raster_crs)
                
                # Filter for land polygons and get their boundaries (the interface)
                # Note: getting the boundary of land inherently gives us the land/water edge
                land_polygons = raster_gdf[raster_gdf['val'] == LAND_VALUE]
                
                if land_polygons.empty:
                    print("  No land found in reference raster for this extent.")
                    continue
                    
                # Convert polygons to boundaries (lines) and project to meters
                ref_interface = land_polygons.geometry.boundary
                ref_interface_meters = ref_interface.to_crs(TARGET_CRS).union_all()
                
                # 5. Buffer Analysis (Intersection)
                # Combine all SAR lines in this shapefile into one geometry for cleaner intersection
                sar_lines_union = gdf.union_all()
                
                for dist in BUFFER_DISTANCES:
                    # Buffer the reference mask's interface
                    ref_buffer = ref_interface_meters.buffer(dist)
                    
                    # Intersect SAR coastline with the reference buffer
                    intersecting_sar = sar_lines_union.intersection(ref_buffer)
                    
                    # Calculate the length of the SAR line that falls inside this buffer
                    intersecting_length = intersecting_sar.length
                    buffer_stats[dist] += intersecting_length
                    
                    percent_in_buffer = (intersecting_length / shp_length) * 100 if shp_length > 0 else 0
                    print(f"  {dist}m Buffer: {percent_in_buffer:.2f}% ({intersecting_length:.2f} m)")
                    
            except ValueError as e:
                print(f"  Skipping raster clip (likely falls outside raster extent): {e}")

    # --- Final Output ---
    print("\n" + "="*50)
    print("FINAL AGGREGATED RESULTS")
    print("="*50)
    print(f"Total files processed: {len(shp_files)}")
    print(f"Collective SAR Coastline Length: {total_sar_length:.2f} meters ({total_sar_length/1000:.2f} km)\n")
    
    print("Percentage of Total SAR Coastline within Reference Land/Water Interface:")
    for dist in BUFFER_DISTANCES:
        total_intersecting = buffer_stats[dist]
        overall_percent = (total_intersecting / total_sar_length) * 100 if total_sar_length > 0 else 0
        print(f"  {dist}m Buffer: {overall_percent:.2f}% (Total matching length: {total_intersecting:.2f} m)")

In [9]:
# --- Configuration ---
SHP_DIR = '/Users/sahuang/Documents/DATA/umbra/ALL_RTC_INPUT/'
REF_RASTER_PATH = '/Users/sahuang/Documents/DATA/am_samoa/Landcover/am_samoa_ccap_ocean_mask_small.tif'
# Define a projected CRS (e.g., a UTM zone) to ensure lengths and buffers are in meters
TARGET_CRS = "EPSG:32702"  # Change this to your local UTM EPSG code!
BUFFER_DISTANCES = [1, 5, 10] # in meters

process_coastlines(shape_dir=SHP_DIR,
                   ref_raster=REF_RASTER_PATH,
                   crs=TARGET_CRS,
                   buffs=BUFFER_DISTANCES)


Processing 20250525_filt_4_cluster_coastline_improved_auto.shp...
  SAR Coastline Length: 18928.58 m


/var/folders/xg/d7l0rjhd61v1zgdny2854dq8w3qvvz/T/ipykernel_69574/1506683650.py:65: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  ref_interface_meters = ref_interface.to_crs(TARGET_CRS).unary_union
/var/folders/xg/d7l0rjhd61v1zgdny2854dq8w3qvvz/T/ipykernel_69574/1506683650.py:69: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  sar_lines_union = gdf.unary_union


  1m Buffer: 4.57% (865.59 m)
  5m Buffer: 22.42% (4243.95 m)
  10m Buffer: 44.83% (8484.88 m)

Processing 20250501_filt_4_cluster_coastline_improved_auto.shp...
  SAR Coastline Length: 18812.87 m
  1m Buffer: 4.55% (855.37 m)


/var/folders/xg/d7l0rjhd61v1zgdny2854dq8w3qvvz/T/ipykernel_69574/1506683650.py:65: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  ref_interface_meters = ref_interface.to_crs(TARGET_CRS).unary_union
/var/folders/xg/d7l0rjhd61v1zgdny2854dq8w3qvvz/T/ipykernel_69574/1506683650.py:69: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  sar_lines_union = gdf.unary_union


  5m Buffer: 23.11% (4347.13 m)
  10m Buffer: 44.81% (8429.13 m)

Processing 20250210_filt_6_cluster_coastline_improved_auto.shp...
  SAR Coastline Length: 20069.19 m


...

  5m Buffer: 28.25% (2728.68 m)
  10m Buffer: 48.47% (4681.84 m)

Processing 20250403_filt_4_cluster_coastline_improved_auto.shp...
  SAR Coastline Length: 25774.59 m


/var/folders/xg/d7l0rjhd61v1zgdny2854dq8w3qvvz/T/ipykernel_69574/1506683650.py:65: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  ref_interface_meters = ref_interface.to_crs(TARGET_CRS).unary_union
/var/folders/xg/d7l0rjhd61v1zgdny2854dq8w3qvvz/T/ipykernel_69574/1506683650.py:69: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  sar_lines_union = gdf.unary_union


  1m Buffer: 6.31% (1627.00 m)
  5m Buffer: 26.91% (6935.14 m)
  10m Buffer: 47.43% (12224.02 m)

Processing 20250321_filt_6_cluster_coastline_improved_auto.shp...
  SAR Coastline Length: 13570.90 m
  1m Buffer: 3.24% (439.80 m)


/var/folders/xg/d7l0rjhd61v1zgdny2854dq8w3qvvz/T/ipykernel_69574/1506683650.py:65: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  ref_interface_meters = ref_interface.to_crs(TARGET_CRS).unary_union
/var/folders/xg/d7l0rjhd61v1zgdny2854dq8w3qvvz/T/ipykernel_69574/1506683650.py:69: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  sar_lines_union = gdf.unary_union


  5m Buffer: 15.34% (2082.39 m)
  10m Buffer: 31.29% (4246.38 m)

FINAL AGGREGATED RESULTS
Total files processed: 23
Collective SAR Coastline Length: 417543.53 meters (417.54 km)

Percentage of Total SAR Coastline within Reference Land/Water Interface:
  1m Buffer: 6.12% (Total matching length: 25544.10 m)
  5m Buffer: 26.29% (Total matching length: 109756.75 m)
  10m Buffer: 45.91% (Total matching length: 191691.76 m)


In [10]:
SHP_DIR = '/Users/sahuang/Documents/DATA/umbra/ALL_SHORELINES_MANUAL/'
REF_RASTER_PATH = '/Users/sahuang/Documents/DATA/am_samoa/Landcover/am_samoa_ccap_ocean_mask_small.tif'
# Define a projected CRS (e.g., a UTM zone) to ensure lengths and buffers are in meters
TARGET_CRS = "EPSG:32702"  # Change this to your local UTM EPSG code!
BUFFER_DISTANCES = [1, 5, 10] # in meters

process_coastlines(shape_dir=SHP_DIR,
                   ref_raster=REF_RASTER_PATH,
                   crs=TARGET_CRS,
                   buffs=BUFFER_DISTANCES)


Processing 20250525_filt_4_cluster_coastline.shp...
  SAR Coastline Length: 6515.57 m
  1m Buffer: 2.92% (190.25 m)


/var/folders/xg/d7l0rjhd61v1zgdny2854dq8w3qvvz/T/ipykernel_69574/1506683650.py:65: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  ref_interface_meters = ref_interface.to_crs(TARGET_CRS).unary_union
/var/folders/xg/d7l0rjhd61v1zgdny2854dq8w3qvvz/T/ipykernel_69574/1506683650.py:69: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  sar_lines_union = gdf.unary_union


  5m Buffer: 15.34% (999.68 m)
  10m Buffer: 34.92% (2274.93 m)

Processing 20250501_filt_4_cluster_coastline.shp...
  SAR Coastline Length: 7144.21 m
  1m Buffer: 4.26% (304.40 m)


/var/folders/xg/d7l0rjhd61v1zgdny2854dq8w3qvvz/T/ipykernel_69574/1506683650.py:65: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  ref_interface_meters = ref_interface.to_crs(TARGET_CRS).unary_union
/var/folders/xg/d7l0rjhd61v1zgdny2854dq8w3qvvz/T/ipykernel_69574/1506683650.py:69: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  sar_lines_union = gdf.unary_union


  5m Buffer: 22.16% (1583.46 m)
  10m Buffer: 47.35% (3382.45 m)

Processing 20250715_filt_merged_coastline.shp...
  SAR Coastline Length: 5633.92 m


...

  5m Buffer: 57.32% (2550.55 m)
  10m Buffer: 79.18% (3523.61 m)

Processing 20241224_filt_6_cluster_coastline.shp...
  SAR Coastline Length: 4682.38 m
  1m Buffer: 10.73% (502.37 m)


/var/folders/xg/d7l0rjhd61v1zgdny2854dq8w3qvvz/T/ipykernel_69574/1506683650.py:65: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  ref_interface_meters = ref_interface.to_crs(TARGET_CRS).unary_union
/var/folders/xg/d7l0rjhd61v1zgdny2854dq8w3qvvz/T/ipykernel_69574/1506683650.py:69: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  sar_lines_union = gdf.unary_union


  5m Buffer: 37.32% (1747.50 m)
  10m Buffer: 54.02% (2529.53 m)

Processing 20250524_filt_4_cluster_coastline.shp...
  SAR Coastline Length: 18328.74 m
  1m Buffer: 12.70% (2328.46 m)


/var/folders/xg/d7l0rjhd61v1zgdny2854dq8w3qvvz/T/ipykernel_69574/1506683650.py:65: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  ref_interface_meters = ref_interface.to_crs(TARGET_CRS).unary_union
/var/folders/xg/d7l0rjhd61v1zgdny2854dq8w3qvvz/T/ipykernel_69574/1506683650.py:69: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  sar_lines_union = gdf.unary_union


  5m Buffer: 43.32% (7940.22 m)
  10m Buffer: 62.14% (11389.70 m)

FINAL AGGREGATED RESULTS
Total files processed: 23
Collective SAR Coastline Length: 157571.52 meters (157.57 km)

Percentage of Total SAR Coastline within Reference Land/Water Interface:
  1m Buffer: 10.37% (Total matching length: 16338.38 m)
  5m Buffer: 41.54% (Total matching length: 65454.63 m)
  10m Buffer: 65.04% (Total matching length: 102483.15 m)
